In [3]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif


# ============================================================
# Portable project-root detection
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent

elif (CURRENT_DIR / "notebooks").exists():
    PROJECT_DIR = CURRENT_DIR

else:
    PROJECT_DIR = None

    for parent in [CURRENT_DIR] + list(CURRENT_DIR.parents):
        if (parent / "notebooks").exists() and (parent / "requirements.txt").exists():
            PROJECT_DIR = parent
            break

    if PROJECT_DIR is None:
        raise FileNotFoundError(
            "Project root could not be detected. "
            "Please start JupyterLab from the CyberXAI-CSE-IDS2018 project folder."
        )


# ============================================================
# Project directories
# ============================================================

TRAIN_DIR = PROJECT_DIR / "data" / "splits" / "train"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
DOCUMENTATION_DIR = PROJECT_DIR / "documentation"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
DOCUMENTATION_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Locate training Parquet files
# ============================================================

train_files = sorted(TRAIN_DIR.glob("*_train.parquet"))

if not train_files:
    raise FileNotFoundError(
        f"No training files were found in:\n{TRAIN_DIR}\n\n"
        "Run Notebook 05 first to create the train/test split."
    )

print("Project directory:", PROJECT_DIR)
print("Training data directory:", TRAIN_DIR)
print("Training files found:", len(train_files))
print("First training file:", train_files[0].name)



Project directory: D:\Sami Data Set\CyberXAI-CSE-IDS2018
Training data directory: D:\Sami Data Set\CyberXAI-CSE-IDS2018\data\splits\train
Training files found: 10
First training file: Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter_train.parquet


In [3]:
schema_columns = pq.ParquetFile(
    train_files[0]
).schema.names

excluded_columns = {
    "original_attack_label",
    "binary_label",
    "source_file_id",
    "source_row_id"
}

predictor_columns = [
    column
    for column in schema_columns
    if column not in excluded_columns
]

print("Total schema columns:", len(schema_columns))
print("Excluded columns:", sorted(excluded_columns))
print("Candidate predictors:", len(predictor_columns))

for number, feature in enumerate(predictor_columns, start=1):
    print(f"{number:02d}. {feature}")

Total schema columns: 81
Excluded columns: ['binary_label', 'original_attack_label', 'source_file_id', 'source_row_id']
Candidate predictors: 77
01. Protocol
02. Flow Duration
03. Total Fwd Packets
04. Total Backward Packets
05. Fwd Packets Length Total
06. Bwd Packets Length Total
07. Fwd Packet Length Max
08. Fwd Packet Length Min
09. Fwd Packet Length Mean
10. Fwd Packet Length Std
11. Bwd Packet Length Max
12. Bwd Packet Length Min
13. Bwd Packet Length Mean
14. Bwd Packet Length Std
15. Flow Bytes/s
16. Flow Packets/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Total
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Total
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd PSH Flags
32. Bwd PSH Flags
33. Fwd URG Flags
34. Bwd URG Flags
35. Fwd Header Length
36. Bwd Header Length
37. Fwd Packets/s
38. Bwd Packets/s
39. Packet Length Min
40. Packet Length Max
41. Packet Length Mean
42. Packet Length

In [4]:
schema = pq.ParquetFile(train_files[0]).schema_arrow

type_records = []

for field in schema:
    if field.name in predictor_columns:
        type_records.append({
            "feature": field.name,
            "parquet_type": str(field.type)
        })

predictor_types = pd.DataFrame(type_records)

display(predictor_types)

predictor_types.to_csv(
    DOCUMENTATION_DIR / "predictor_data_types.csv",
    index=False
)



,feature,parquet_type
0,Protocol,int8
1,Flow Duration,int32
2,Total Fwd Packets,int32
3,Total Backward Packets,int32
4,Fwd Packets Length Total,int32
...,...,...
72,Active Min,float
73,Idle Mean,float
74,Idle Std,float
75,Idle Max,float


In [5]:
sample_parts = []

RECORDS_PER_CLASS_PER_FILE = 5_000
RANDOM_SEED = 42

sample_columns = predictor_columns + ["binary_label"]

for file_number, file_path in enumerate(train_files, start=1):
    print(
        f"[{file_number}/{len(train_files)}] "
        f"Sampling {file_path.name}",
        flush=True
    )

    df = pd.read_parquet(
        file_path,
        columns=sample_columns
    )

    for class_value in [0, 1]:
        class_data = df[
            df["binary_label"] == class_value
        ]

        sample_size = min(
            RECORDS_PER_CLASS_PER_FILE,
            len(class_data)
        )

        if sample_size > 0:
            selected = class_data.sample(
                n=sample_size,
                random_state=(
                    RANDOM_SEED
                    + file_number * 10
                    + class_value
                )
            )

            sample_parts.append(selected)

    del df
    gc.collect()

feature_selection_sample = pd.concat(
    sample_parts,
    ignore_index=True
)

sample_path = (
    INTERIM_DIR
    / "training_feature_selection_sample.parquet"
)

feature_selection_sample.to_parquet(
    sample_path,
    engine="pyarrow",
    compression="snappy",
    index=False
)

print(
    "Feature-selection sample rows:",
    f"{len(feature_selection_sample):,}"
)

display(
    feature_selection_sample["binary_label"]
    .value_counts()
    .rename_axis("binary_label")
    .reset_index(name="count")
)

print("Sample saved to:", sample_path)


[1/10] Sampling Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter_train.parquet
[2/10] Sampling Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter_train.parquet
[3/10] Sampling DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter_train.parquet
[4/10] Sampling DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter_train.parquet
[5/10] Sampling DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter_train.parquet
[6/10] Sampling DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter_train.parquet
[7/10] Sampling Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter_train.parquet
[8/10] Sampling Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter_train.parquet
[9/10] Sampling Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter_train.parquet
[10/10] Sampling Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter_train.parquet
Feature-selection sample rows: 90,706


,binary_label,count
0,0,50000
1,1,40706


Sample saved to: D:\Sami Data Set\data\interim\training_feature_selection_sample.parquet


In [6]:
X_sample = feature_selection_sample[
    predictor_columns
].copy()

y_sample = feature_selection_sample[
    "binary_label"
].astype("int8").copy()

# Convert all predictors to numeric
for column in X_sample.columns:
    X_sample[column] = pd.to_numeric(
        X_sample[column],
        errors="coerce"
    )

print("Predictor matrix:", X_sample.shape)
print("Target records:", len(y_sample))
print("Missing predictor values:", int(X_sample.isna().sum().sum()))

Predictor matrix: (90706, 77)
Target records: 90706
Missing predictor values: 0


In [7]:
screening_records = []

for feature in X_sample.columns:
    values = X_sample[feature]

    unique_count = int(values.nunique(dropna=False))
    variance = float(values.var(ddof=0))

    value_frequencies = values.value_counts(
        normalize=True,
        dropna=False
    )

    dominant_value_percentage = (
        float(value_frequencies.iloc[0] * 100)
        if not value_frequencies.empty
        else np.nan
    )

    screening_records.append({
        "feature": feature,
        "unique_values": unique_count,
        "variance": variance,
        "dominant_value_percentage":
            dominant_value_percentage,
        "is_constant": unique_count <= 1,
        "is_near_constant":
            dominant_value_percentage >= 99.9
    })

variance_screening = pd.DataFrame(
    screening_records
).sort_values(
    ["is_constant", "is_near_constant", "variance"],
    ascending=[False, False, True]
)

display(variance_screening)

variance_screening.to_csv(
    TABLES_DIR / "variance_feature_screening.csv",
    index=False
)



,feature,unique_values,variance,dominant_value_percentage,is_constant,is_near_constant
31,Bwd PSH Flags,1,0.000000e+00,100.000000,True,True
33,Bwd URG Flags,1,0.000000e+00,100.000000,True,True
55,Fwd Avg Bytes/Bulk,1,0.000000e+00,100.000000,True,True
56,Fwd Avg Packets/Bulk,1,0.000000e+00,100.000000,True,True
57,Fwd Avg Bulk Rate,1,0.000000e+00,100.000000,True,True
...,...,...,...,...,...,...
23,Fwd IAT Max,52703,3.206079e+14,14.895376,False,False
18,Flow IAT Max,57195,3.229451e+14,1.085926,False,False
25,Bwd IAT Total,43369,6.731608e+14,37.772584,False,False
20,Fwd IAT Total,53934,9.527715e+14,14.895376,False,False


In [8]:
constant_features = variance_screening.loc[
    variance_screening["is_constant"],
    "feature"
].tolist()

near_constant_features = variance_screening.loc[
    variance_screening["is_near_constant"]
    & ~variance_screening["is_constant"],
    "feature"
].tolist()

print("Constant features:", constant_features)
print("Near-constant features:", near_constant_features)



Constant features: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Near-constant features: ['Fwd URG Flags', 'CWE Flag Count']


In [9]:
screened_predictors = [
    feature
    for feature in predictor_columns
    if feature not in constant_features
]

X_screened = X_sample[screened_predictors].copy()

print("Predictors before screening:", len(predictor_columns))
print("Constant features removed:", len(constant_features))
print("Predictors after screening:", len(screened_predictors))

Predictors before screening: 77
Constant features removed: 8
Predictors after screening: 69


In [10]:
correlation_matrix = X_screened.corr(
    method="pearson"
)

correlation_matrix.to_csv(
    TABLES_DIR / "training_feature_correlation_matrix.csv"
)

correlated_pair_records = []

columns = correlation_matrix.columns

for first_index in range(len(columns)):
    for second_index in range(first_index + 1, len(columns)):
        correlation_value = correlation_matrix.iloc[
            first_index,
            second_index
        ]

        if abs(correlation_value) >= 0.95:
            correlated_pair_records.append({
                "feature_1": columns[first_index],
                "feature_2": columns[second_index],
                "pearson_correlation":
                    float(correlation_value),
                "absolute_correlation":
                    float(abs(correlation_value))
            })

high_correlation_pairs = pd.DataFrame(
    correlated_pair_records,
    columns=[
        "feature_1",
        "feature_2",
        "pearson_correlation",
        "absolute_correlation"
    ]
).sort_values(
    "absolute_correlation",
    ascending=False
)

display(high_correlation_pairs)

high_correlation_pairs.to_csv(
    TABLES_DIR / "highly_correlated_feature_pairs.csv",
    index=False
)

print(
    "Feature pairs with |correlation| ≥ 0.95:",
    len(high_correlation_pairs)
)

,feature_1,feature_2,pearson_correlation,absolute_correlation
3,Total Fwd Packets,Subflow Fwd Packets,1.000000,1.000000
8,Total Backward Packets,Subflow Bwd Packets,1.000000,1.000000
12,Fwd Packets Length Total,Subflow Fwd Bytes,1.000000,1.000000
19,Bwd Packet Length Mean,Avg Bwd Segment Size,1.000000,1.000000
35,RST Flag Count,ECE Flag Count,1.000000,1.000000
27,Fwd URG Flags,CWE Flag Count,1.000000,1.000000
26,Fwd PSH Flags,SYN Flag Count,1.000000,1.000000
16,Bwd Packets Length Total,Subflow Bwd Bytes,1.000000,1.000000
18,Fwd Packet Length Mean,Avg Fwd Segment Size,1.000000,1.000000
37,Subflow Fwd Packets,Fwd Act Data Packets,0.999741,0.999741


Feature pairs with |correlation| ≥ 0.95: 43


In [11]:
target_correlation_records = []

for feature in X_screened.columns:
    correlation_value = X_screened[feature].corr(
        y_sample
    )

    target_correlation_records.append({
        "feature": feature,
        "target_correlation": float(correlation_value),
        "absolute_target_correlation":
            float(abs(correlation_value))
    })

target_correlation = pd.DataFrame(
    target_correlation_records
).sort_values(
    "absolute_target_correlation",
    ascending=False
)

display(target_correlation.head(20))

target_correlation.to_csv(
    TABLES_DIR / "feature_target_correlation.csv",
    index=False
)

,feature,target_correlation,absolute_target_correlation
60,Fwd Seg Size Min,0.329284,0.329284
58,Init Bwd Win Bytes,-0.279622,0.279622
6,Fwd Packet Length Max,-0.185445,0.185445
11,Bwd Packet Length Min,-0.178516,0.178516
0,Protocol,-0.177827,0.177827
9,Fwd Packet Length Std,-0.171508,0.171508
51,Avg Fwd Segment Size,-0.149496,0.149496
8,Fwd Packet Length Mean,-0.149496,0.149496
43,RST Flag Count,0.140447,0.140447
48,ECE Flag Count,0.140447,0.140447


In [12]:
print("Calculating mutual information...", flush=True)

mutual_information_values = mutual_info_classif(
    X_screened,
    y_sample,
    discrete_features="auto",
    random_state=RANDOM_SEED,
    n_neighbors=3
)

mutual_information_ranking = pd.DataFrame({
    "feature": X_screened.columns,
    "mutual_information":
        mutual_information_values
}).sort_values(
    "mutual_information",
    ascending=False
)

display(mutual_information_ranking.head(20))

mutual_information_ranking.to_csv(
    TABLES_DIR / "mutual_information_ranking.csv",
    index=False
)

Calculating mutual information...


,feature,mutual_information
56,Subflow Bwd Bytes,0.318649
5,Bwd Packets Length Total,0.318337
12,Bwd Packet Length Mean,0.314208
52,Avg Bwd Segment Size,0.313390
10,Bwd Packet Length Max,0.311118
54,Subflow Fwd Bytes,0.310810
4,Fwd Packets Length Total,0.309751
51,Avg Fwd Segment Size,0.304490
8,Fwd Packet Length Mean,0.303807
6,Fwd Packet Length Max,0.302880


In [13]:
print("Training feature-ranking Random Forest...", flush=True)

feature_ranking_forest = RandomForestClassifier(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

feature_ranking_forest.fit(
    X_screened,
    y_sample
)

random_forest_ranking = pd.DataFrame({
    "feature": X_screened.columns,
    "random_forest_importance":
        feature_ranking_forest.feature_importances_
}).sort_values(
    "random_forest_importance",
    ascending=False
)

display(random_forest_ranking.head(20))

random_forest_ranking.to_csv(
    TABLES_DIR / "random_forest_feature_ranking.csv",
    index=False
)

Training feature-ranking Random Forest...


,feature,random_forest_importance
6,Fwd Packet Length Max,0.049539
58,Init Bwd Win Bytes,0.042965
57,Init Fwd Win Bytes,0.040374
24,Fwd IAT Min,0.040044
9,Fwd Packet Length Std,0.036900
60,Fwd Seg Size Min,0.034706
16,Flow IAT Mean,0.027849
51,Avg Fwd Segment Size,0.027573
19,Flow IAT Min,0.026126
54,Subflow Fwd Bytes,0.025986


In [14]:
print("Training feature-ranking Random Forest...", flush=True)

feature_ranking_forest = RandomForestClassifier(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

feature_ranking_forest.fit(
    X_screened,
    y_sample
)

random_forest_ranking = pd.DataFrame({
    "feature": X_screened.columns,
    "random_forest_importance":
        feature_ranking_forest.feature_importances_
}).sort_values(
    "random_forest_importance",
    ascending=False
)

display(random_forest_ranking.head(20))

random_forest_ranking.to_csv(
    TABLES_DIR / "random_forest_feature_ranking.csv",
    index=False
)

Training feature-ranking Random Forest...


,feature,random_forest_importance
6,Fwd Packet Length Max,0.049539
58,Init Bwd Win Bytes,0.042965
57,Init Fwd Win Bytes,0.040374
24,Fwd IAT Min,0.040044
9,Fwd Packet Length Std,0.036900
60,Fwd Seg Size Min,0.034706
16,Flow IAT Mean,0.027849
51,Avg Fwd Segment Size,0.027573
19,Flow IAT Min,0.026126
54,Subflow Fwd Bytes,0.025986


In [15]:
combined_ranking = (
    target_correlation
    .merge(
        mutual_information_ranking,
        on="feature",
        how="inner"
    )
    .merge(
        random_forest_ranking,
        on="feature",
        how="inner"
    )
)

feature_count = len(combined_ranking)

combined_ranking[
    "target_correlation_rank"
] = combined_ranking[
    "absolute_target_correlation"
].rank(
    ascending=False,
    method="average"
)

combined_ranking[
    "mutual_information_rank"
] = combined_ranking[
    "mutual_information"
].rank(
    ascending=False,
    method="average"
)

combined_ranking[
    "random_forest_rank"
] = combined_ranking[
    "random_forest_importance"
].rank(
    ascending=False,
    method="average"
)

combined_ranking["mean_rank"] = combined_ranking[
    [
        "target_correlation_rank",
        "mutual_information_rank",
        "random_forest_rank"
    ]
].mean(axis=1)

combined_ranking = combined_ranking.sort_values(
    "mean_rank",
    ascending=True
).reset_index(drop=True)

combined_ranking["consensus_position"] = (
    np.arange(1, len(combined_ranking) + 1)
)

display(combined_ranking.head(25))

combined_ranking.to_csv(
    TABLES_DIR / "consensus_feature_ranking_initial.csv",
    index=False
)


,feature,target_correlation,absolute_target_correlation,mutual_information,random_forest_importance,target_correlation_rank,mutual_information_rank,random_forest_rank,mean_rank,consensus_position
0,Fwd Packet Length Max,-0.185445,0.185445,0.302880,0.049539,3.0,10.0,1.0,4.666667,1
1,Init Bwd Win Bytes,-0.279622,0.279622,0.247688,0.042965,2.0,14.0,2.0,6.000000,2
2,Fwd Packet Length Std,-0.171508,0.171508,0.290548,0.036900,6.0,12.0,5.0,7.666667,3
3,Avg Fwd Segment Size,-0.149496,0.149496,0.304490,0.027573,7.5,8.0,8.0,7.833333,4
4,Fwd Packet Length Mean,-0.149496,0.149496,0.303807,0.023523,7.5,9.0,13.0,9.833333,5
5,Init Fwd Win Bytes,0.086761,0.086761,0.241429,0.040374,20.0,16.0,3.0,13.000000,6
6,Fwd Seg Size Min,0.329284,0.329284,0.066520,0.034706,1.0,46.0,6.0,17.666667,7
7,Flow Duration,-0.112419,0.112419,0.195867,0.022911,15.0,26.0,14.0,18.333333,8
8,Flow IAT Max,-0.084525,0.084525,0.211979,0.023854,22.0,22.0,12.0,18.666667,9
9,Packet Length Variance,-0.088502,0.088502,0.242991,0.016320,19.0,15.0,26.0,20.000000,10


In [16]:
consensus_rank_lookup = dict(
    zip(
        combined_ranking["feature"],
        combined_ranking["mean_rank"]
    )
)

features_to_remove_for_correlation = set()
correlation_decision_records = []

for _, row in high_correlation_pairs.iterrows():
    feature_1 = row["feature_1"]
    feature_2 = row["feature_2"]

    if (
        feature_1 in features_to_remove_for_correlation
        or feature_2 in features_to_remove_for_correlation
    ):
        continue

    rank_1 = consensus_rank_lookup[feature_1]
    rank_2 = consensus_rank_lookup[feature_2]

    if rank_1 <= rank_2:
        retained_feature = feature_1
        removed_feature = feature_2
    else:
        retained_feature = feature_2
        removed_feature = feature_1

    features_to_remove_for_correlation.add(
        removed_feature
    )

    correlation_decision_records.append({
        "feature_1": feature_1,
        "feature_2": feature_2,
        "absolute_correlation":
            row["absolute_correlation"],
        "retained_feature": retained_feature,
        "removed_feature": removed_feature,
        "decision_basis":
            "Better combined feature-selection rank"
    })

correlation_decisions = pd.DataFrame(
    correlation_decision_records
)

display(correlation_decisions)

correlation_decisions.to_csv(
    TABLES_DIR / "correlation_reduction_decisions.csv",
    index=False
)

,feature_1,feature_2,absolute_correlation,retained_feature,removed_feature,decision_basis
0,Total Fwd Packets,Subflow Fwd Packets,1.000000,Subflow Fwd Packets,Total Fwd Packets,Better combined feature-selection rank
1,Total Backward Packets,Subflow Bwd Packets,1.000000,Subflow Bwd Packets,Total Backward Packets,Better combined feature-selection rank
2,Fwd Packets Length Total,Subflow Fwd Bytes,1.000000,Subflow Fwd Bytes,Fwd Packets Length Total,Better combined feature-selection rank
3,Bwd Packet Length Mean,Avg Bwd Segment Size,1.000000,Avg Bwd Segment Size,Bwd Packet Length Mean,Better combined feature-selection rank
4,RST Flag Count,ECE Flag Count,1.000000,RST Flag Count,ECE Flag Count,Better combined feature-selection rank
5,Fwd URG Flags,CWE Flag Count,1.000000,Fwd URG Flags,CWE Flag Count,Better combined feature-selection rank
6,Fwd PSH Flags,SYN Flag Count,1.000000,Fwd PSH Flags,SYN Flag Count,Better combined feature-selection rank
7,Bwd Packets Length Total,Subflow Bwd Bytes,1.000000,Bwd Packets Length Total,Subflow Bwd Bytes,Better combined feature-selection rank
8,Fwd Packet Length Mean,Avg Fwd Segment Size,1.000000,Avg Fwd Segment Size,Fwd Packet Length Mean,Better combined feature-selection rank
9,Subflow Fwd Packets,Fwd Act Data Packets,0.999741,Subflow Fwd Packets,Fwd Act Data Packets,Better combined feature-selection rank


In [17]:
final_consensus_ranking = combined_ranking[
    ~combined_ranking["feature"].isin(
        features_to_remove_for_correlation
    )
].copy()

final_consensus_ranking = (
    final_consensus_ranking
    .sort_values("mean_rank")
    .reset_index(drop=True)
)

final_consensus_ranking[
    "final_consensus_position"
] = np.arange(
    1,
    len(final_consensus_ranking) + 1
)

display(final_consensus_ranking.head(25))

final_consensus_ranking.to_csv(
    TABLES_DIR / "final_consensus_feature_ranking.csv",
    index=False
)

,feature,target_correlation,absolute_target_correlation,mutual_information,random_forest_importance,target_correlation_rank,mutual_information_rank,random_forest_rank,mean_rank,consensus_position,final_consensus_position
0,Fwd Packet Length Max,-0.185445,0.185445,0.302880,0.049539,3.0,10.0,1.0,4.666667,1,1
1,Init Bwd Win Bytes,-0.279622,0.279622,0.247688,0.042965,2.0,14.0,2.0,6.000000,2,2
2,Fwd Packet Length Std,-0.171508,0.171508,0.290548,0.036900,6.0,12.0,5.0,7.666667,3,3
3,Avg Fwd Segment Size,-0.149496,0.149496,0.304490,0.027573,7.5,8.0,8.0,7.833333,4,4
4,Init Fwd Win Bytes,0.086761,0.086761,0.241429,0.040374,20.0,16.0,3.0,13.000000,6,5
5,Fwd Seg Size Min,0.329284,0.329284,0.066520,0.034706,1.0,46.0,6.0,17.666667,7,6
6,Flow Duration,-0.112419,0.112419,0.195867,0.022911,15.0,26.0,14.0,18.333333,8,7
7,Flow IAT Max,-0.084525,0.084525,0.211979,0.023854,22.0,22.0,12.0,18.666667,9,8
8,Packet Length Variance,-0.088502,0.088502,0.242991,0.016320,19.0,15.0,26.0,20.000000,10,9
9,Subflow Fwd Bytes,0.027906,0.027906,0.310810,0.025986,44.5,6.0,10.0,20.166667,11,10


In [18]:
top_10_features = final_consensus_ranking[
    "feature"
].head(10).tolist()

top_15_features = final_consensus_ranking[
    "feature"
].head(15).tolist()

top_20_features = final_consensus_ranking[
    "feature"
].head(20).tolist()

feature_sets = {
    "top_10": top_10_features,
    "top_15": top_15_features,
    "top_20": top_20_features,
    "constant_features_removed": constant_features,
    "correlated_features_removed": sorted(
        features_to_remove_for_correlation
    )
}

with open(
    DOCUMENTATION_DIR / "candidate_feature_sets.json",
    "w",
    encoding="utf-8"
) as feature_file:
    json.dump(
        feature_sets,
        feature_file,
        indent=4
    )

print("Top-10 features:")
for feature in top_10_features:
    print("-", feature)

print("\nTop-15 features:")
for feature in top_15_features:
    print("-", feature)

print("\nTop-20 features:")
for feature in top_20_features:
    print("-", feature)

Top-10 features:
- Fwd Packet Length Max
- Init Bwd Win Bytes
- Fwd Packet Length Std
- Avg Fwd Segment Size
- Init Fwd Win Bytes
- Fwd Seg Size Min
- Flow Duration
- Flow IAT Max
- Packet Length Variance
- Subflow Fwd Bytes

Top-15 features:
- Fwd Packet Length Max
- Init Bwd Win Bytes
- Fwd Packet Length Std
- Avg Fwd Segment Size
- Init Fwd Win Bytes
- Fwd Seg Size Min
- Flow Duration
- Flow IAT Max
- Packet Length Variance
- Subflow Fwd Bytes
- Flow IAT Mean
- Packet Length Mean
- Packet Length Max
- Avg Bwd Segment Size
- Fwd IAT Std

Top-20 features:
- Fwd Packet Length Max
- Init Bwd Win Bytes
- Fwd Packet Length Std
- Avg Fwd Segment Size
- Init Fwd Win Bytes
- Fwd Seg Size Min
- Flow Duration
- Flow IAT Max
- Packet Length Variance
- Subflow Fwd Bytes
- Flow IAT Mean
- Packet Length Mean
- Packet Length Max
- Avg Bwd Segment Size
- Fwd IAT Std
- Bwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packets Length Total
- Bwd IAT Min
- RST Flag Count


In [19]:
top_10_features = final_consensus_ranking[
    "feature"
].head(10).tolist()

top_15_features = final_consensus_ranking[
    "feature"
].head(15).tolist()

top_20_features = final_consensus_ranking[
    "feature"
].head(20).tolist()

feature_sets = {
    "top_10": top_10_features,
    "top_15": top_15_features,
    "top_20": top_20_features,
    "constant_features_removed": constant_features,
    "correlated_features_removed": sorted(
        features_to_remove_for_correlation
    )
}

with open(
    DOCUMENTATION_DIR / "candidate_feature_sets.json",
    "w",
    encoding="utf-8"
) as feature_file:
    json.dump(
        feature_sets,
        feature_file,
        indent=4
    )

print("Top-10 features:")
for feature in top_10_features:
    print("-", feature)

print("\nTop-15 features:")
for feature in top_15_features:
    print("-", feature)

print("\nTop-20 features:")
for feature in top_20_features:
    print("-", feature)

Top-10 features:
- Fwd Packet Length Max
- Init Bwd Win Bytes
- Fwd Packet Length Std
- Avg Fwd Segment Size
- Init Fwd Win Bytes
- Fwd Seg Size Min
- Flow Duration
- Flow IAT Max
- Packet Length Variance
- Subflow Fwd Bytes

Top-15 features:
- Fwd Packet Length Max
- Init Bwd Win Bytes
- Fwd Packet Length Std
- Avg Fwd Segment Size
- Init Fwd Win Bytes
- Fwd Seg Size Min
- Flow Duration
- Flow IAT Max
- Packet Length Variance
- Subflow Fwd Bytes
- Flow IAT Mean
- Packet Length Mean
- Packet Length Max
- Avg Bwd Segment Size
- Fwd IAT Std

Top-20 features:
- Fwd Packet Length Max
- Init Bwd Win Bytes
- Fwd Packet Length Std
- Avg Fwd Segment Size
- Init Fwd Win Bytes
- Fwd Seg Size Min
- Flow Duration
- Flow IAT Max
- Packet Length Variance
- Subflow Fwd Bytes
- Flow IAT Mean
- Packet Length Mean
- Packet Length Max
- Avg Bwd Segment Size
- Fwd IAT Std
- Bwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packets Length Total
- Bwd IAT Min
- RST Flag Count


In [20]:
feature_set_table = pd.DataFrame({
    "position": range(1, 21),
    "top_10": (
        top_10_features
        + [""] * (20 - len(top_10_features))
    ),
    "top_15": (
        top_15_features
        + [""] * (20 - len(top_15_features))
    ),
    "top_20": top_20_features
})

display(feature_set_table)

feature_set_table.to_csv(
    TABLES_DIR / "candidate_feature_sets.csv",
    index=False
)




,position,top_10,top_15,top_20
0,1,Fwd Packet Length Max,Fwd Packet Length Max,Fwd Packet Length Max
1,2,Init Bwd Win Bytes,Init Bwd Win Bytes,Init Bwd Win Bytes
2,3,Fwd Packet Length Std,Fwd Packet Length Std,Fwd Packet Length Std
3,4,Avg Fwd Segment Size,Avg Fwd Segment Size,Avg Fwd Segment Size
4,5,Init Fwd Win Bytes,Init Fwd Win Bytes,Init Fwd Win Bytes
5,6,Fwd Seg Size Min,Fwd Seg Size Min,Fwd Seg Size Min
6,7,Flow Duration,Flow Duration,Flow Duration
7,8,Flow IAT Max,Flow IAT Max,Flow IAT Max
8,9,Packet Length Variance,Packet Length Variance,Packet Length Variance
9,10,Subflow Fwd Bytes,Subflow Fwd Bytes,Subflow Fwd Bytes


## Feature-Selection Conclusion

Feature selection was conducted exclusively using records from the
training partition. The frozen test set was not accessed during
variance screening, correlation analysis, mutual-information
calculation or Random Forest feature ranking.

Constant features were excluded because they contained no variation.
Near-constant features were recorded for review rather than removed
automatically. Highly correlated feature pairs were identified using
an absolute Pearson correlation threshold of 0.95. Where redundancy
was identified, the feature with the stronger combined ranking was
retained.

The final consensus ranking combines absolute feature-to-target
correlation, mutual information and Random Forest feature importance.
Candidate top-10, top-15 and top-20 feature sets were generated for
subsequent validation. No feature set has yet been declared optimal;
the candidate subsets must be compared through the same training and
validation procedures before a final set is selected.